In [1]:
# this notebook is for extracting speech chunks from our 24/7 recording
#from joblib import delayed, Parallel
from glob import glob
from silero_vad import load_silero_vad, read_audio, get_speech_timestamps
from neo.io import BlackrockIO
import torch
import torchaudio
import pandas as pd
import os
import numpy as np
import json
import soundfile as sf
from pyNsXStitch.stitchers import StitchedNeVFile, StitchedNsXFile
from pyNsXStitch.helpers import find_nsx_in_range, get_nsx_start_timestamp, get_nsx_duration
from datetime import datetime, timedelta

In [2]:
# set dirs
read_dir = '/mnt/datalake/data/emu'
write_dir = '/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new'

In [23]:
# set consts
patient = 'YFA'
original_fs = 30_000
target_fs = 16_000

In [24]:
os.makedirs(write_dir, exist_ok=True)
os.makedirs(f"{write_dir}/{patient}", exist_ok=True)

In [25]:
# get recording dirs
recording_dirs = glob(f'{read_dir}/{patient}Datafile/DATA/2024*')
# sort by timestamp
recording_dirs = sorted(recording_dirs, key=lambda a: datetime.strptime(os.path.basename(a), "%Y%m%d-%H%M%S"))

# for YFF, do not keep first 3 toc
#recording_dirs = recording_dirs[3:]


In [26]:
# get all nsp1 files
nsp1_files = []
for idx, dir_ in enumerate(recording_dirs):
    files = glob(f'{dir_}/NSP1*.ns5')
    files = sorted(files, key=lambda a: int(a[-7:-4]))
    nsp1_files.extend(files)

In [27]:
import os
import subprocess
from pathlib import Path

# Adjust these for your cluster
partition = "guppy"            # e.g. "gpu", "a100", "l40", etc.
conda_activate = "source /scratch/tahaismail424/.bash_profile && conda activate speech_247_env"
patient_dir = Path(f"/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/{patient}")
logs_dir = Path(f"/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/{patient}/slurm_logs")
batch_dir = Path(f"/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/{patient}/slurm_scripts")
out_dir = Path(f"/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/{patient}/vad_outs")

logs_dir.mkdir(parents=True, exist_ok=True)
batch_dir.mkdir(parents=True, exist_ok=True)
out_dir.mkdir(parents=True, exist_ok=True)

worker_path = "/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/vad_worker.py"

submitted = []
failed_submissions = []

for ns5_file in nsp1_files:
    chunk_name = Path(ns5_file).stem
    job_name = f"vad_{chunk_name}"  # keep it reasonably short

    sbatch_text = f"""#!/bin/bash
#SBATCH --job-name={job_name}
#SBATCH --partition={partition}
#SBATCH --gres=mps:l40:10
#SBATCH --cpus-per-task=4
#SBATCH --mem=24G
#SBATCH --qos=default_tier
##SBATCH --time=04:00:00
#SBATCH --output={logs_dir}/{chunk_name}_%j.out
#SBATCH --error={logs_dir}/{chunk_name}_%j.err

set -eo pipefail

{conda_activate}

echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "FILE: {ns5_file}"

python {worker_path} "{ns5_file}" "{write_dir}"

echo "END: $(date)"
"""

    sbatch_path = batch_dir / f"{chunk_name}.sbatch"
    sbatch_path.write_text(sbatch_text)

    try:
        res = subprocess.run(
            ["sbatch", str(sbatch_path)],
            capture_output=True,
            text=True,
            check=True
        )
        submitted.append((ns5_file, res.stdout.strip()))
        print(f"submitted: {chunk_name} -> {res.stdout.strip()}")
    except subprocess.CalledProcessError as e:
        failed_submissions.append((ns5_file, e.stderr.strip()))
        print(f"FAILED SUBMISSION: {chunk_name}")
        print(e.stderr)

KeyboardInterrupt: 

In [28]:
# load all csvs
import pandas as pd
csv_list = glob(f"{str(out_dir)}/*.csv") 
csvs = []
for path in csv_list:
    try:
        df = pd.read_csv(path, parse_dates=['utc_ts', 'utc_te'])
        csvs.append(df)
    except Exception as e:
        print(e)
all_data = pd.concat(csvs, ignore_index=True)

No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns to parse from file
No columns

In [29]:
import pandas as pd

def merge_utc_br_intervals_by_toc(
    df: pd.DataFrame,
    patient_col: str = "patient",
    toc_col: str = "toc",
    start_col: str = "utc_ts",
    end_col: str = "utc_te",
    br_start_col: str = "br_ts",
    br_end_col: str = "br_te",
    max_gap: str | pd.Timedelta = "1min",
    max_len: str | pd.Timedelta = "30min",
) -> pd.DataFrame:
    """
    Merge intervals within each (patient, TOC) only, while preserving both UTC and
    Blackrock timestamp/sample boundaries.

    Returns
    -------
    pd.DataFrame
        Columns:
        - patient
        - toc
        - interval_idx_within_toc
        - interval_id  (patient + toc + idx)
        - utc_ts
        - utc_te
        - br_ts
        - br_te
        - n_intervals
        - duration_s
        - br_duration
    """
    if df.empty:
        return pd.DataFrame(
            columns=[
                patient_col,
                toc_col,
                "interval_idx_within_toc",
                "interval_id",
                start_col,
                end_col,
                br_start_col,
                br_end_col,
                "n_intervals",
                "duration_s",
                "br_duration",
            ]
        )

    max_gap = pd.Timedelta(max_gap)
    max_len = pd.Timedelta(max_len)

    work = df[[patient_col, toc_col, start_col, end_col, br_start_col, br_end_col]].copy()
    work[start_col] = pd.to_datetime(work[start_col])
    work[end_col] = pd.to_datetime(work[end_col])
    work = work.sort_values([patient_col, toc_col, start_col, end_col]).reset_index(drop=True)

    merged_rows = []

    # group by BOTH patient and toc
    for (patient, toc), group in work.groupby([patient_col, toc_col], sort=False):
        group = group.reset_index(drop=True)

        current_start = group.loc[0, start_col]
        current_end = group.loc[0, end_col]
        current_br_start = group.loc[0, br_start_col]
        current_br_end = group.loc[0, br_end_col]
        count = 1
        interval_idx = 0

        for i in range(1, len(group)):
            start = group.loc[i, start_col]
            end = group.loc[i, end_col]
            br_start = group.loc[i, br_start_col]
            br_end = group.loc[i, br_end_col]

            gap = start - current_end
            proposed_end = max(current_end, end)
            proposed_len = proposed_end - current_start

            if gap <= max_gap and proposed_len <= max_len:
                current_end = end
                current_br_end = br_end
                count += 1
            else:
                merged_rows.append({
                    patient_col: patient,
                    toc_col: toc,
                    "interval_idx_within_toc": interval_idx,
                    "interval_id": f"{toc}_{interval_idx:04d}",
                    start_col: current_start,
                    end_col: current_end,
                    br_start_col: current_br_start,
                    br_end_col: current_br_end,
                    "n_intervals": count,
                })
                interval_idx += 1

                current_start = start
                current_end = end
                current_br_start = br_start
                current_br_end = br_end
                count = 1

        # final interval
        merged_rows.append({
            patient_col: patient,
            toc_col: toc,
            "interval_idx_within_toc": interval_idx,
            "interval_id": f"{toc}_{interval_idx:04d}",
            start_col: current_start,
            end_col: current_end,
            br_start_col: current_br_start,
            br_end_col: current_br_end,
            "n_intervals": count,
        })

    merged_df = pd.DataFrame(merged_rows)

    merged_df["duration_s"] = (merged_df[end_col] - merged_df[start_col]).dt.total_seconds()
    merged_df["br_duration"] = merged_df[br_end_col] - merged_df[br_start_col]

    return merged_df

In [32]:
# get merged df
merged_df = merge_utc_br_intervals_by_toc(
    all_data,
    toc_col="toc",
    start_col="utc_ts",
    end_col="utc_te",
    br_start_col="br_ts",
    br_end_col="br_te",
    max_gap="1min",
    max_len="30min",
)

In [30]:
# save merged df for workers to access
manifest_path = f"{write_dir}/{patient}/{patient}_vad_merged_intervals.csv"
merged_df.to_csv(manifest_path)

KeyboardInterrupt: 

In [31]:
import subprocess
from pathlib import Path

partition = "Universal"   # or whatever partition you use
conda_activate = "source /scratch/tahaismail424/.bash_profile && conda activate speech_247_env"

patient_dir = Path(f"/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/{patient}")
logs_dir = patient_dir / "stitch_slurm_logs"
batch_dir = patient_dir / "stitch_slurm_scripts"
logs_dir.mkdir(parents=True, exist_ok=True)
batch_dir.mkdir(parents=True, exist_ok=True)

worker_path = "/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/stitch_worker.py"

submitted = []
failed = []

for row_idx, row in merged_df.iterrows():
    interval_id = row["interval_id"]
    out_dir = Path(write_dir) / patient / "vad_data" / interval_id
    done_path = out_dir / "_SUCCESS"

    if done_path.exists():
        print(f"skip existing: {interval_id}")
        continue

    sbatch_text = f"""#!/bin/bash
#SBATCH --job-name=stitch_{interval_id}
#SBATCH --partition={partition}
#SBATCH --cpus-per-task=2
#SBATCH --mem=16G
#SBATCH --output={logs_dir}/{interval_id}_%j.out
#SBATCH --error={logs_dir}/{interval_id}_%j.err
#SBATCH --qos=default_tier

set -eo pipefail

{conda_activate}

echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "INTERVAL_ID: {interval_id}"
echo "ROW_IDX: {row_idx}"

python {worker_path} "{manifest_path}" "{row_idx}" "{write_dir}"

echo "END: $(date)"
"""

    sbatch_path = batch_dir / f"{interval_id}.sbatch"
    sbatch_path.write_text(sbatch_text)

    try:
        res = subprocess.run(
            ["sbatch", str(sbatch_path)],
            capture_output=True,
            text=True,
            check=True
        )
        submitted.append((interval_id, res.stdout.strip()))
        print(f"submitted: {interval_id} -> {res.stdout.strip()}")
    except subprocess.CalledProcessError as e:
        failed.append((interval_id, e.stderr.strip()))
        print(f"FAILED SUBMISSION: {interval_id}")
        print(e.stderr)

print(f"submitted={len(submitted)}, failed={len(failed)}")

skip existing: 20240423-124841_0000
skip existing: 20240423-124841_0001
skip existing: 20240423-124841_0002
skip existing: 20240423-124841_0003
skip existing: 20240423-124841_0004
skip existing: 20240423-124841_0005
skip existing: 20240423-124841_0006
skip existing: 20240423-124841_0007
skip existing: 20240423-124841_0008
skip existing: 20240423-124841_0009
skip existing: 20240423-124841_0010
skip existing: 20240423-124841_0011
skip existing: 20240423-124841_0012
skip existing: 20240423-171352_0000
skip existing: 20240423-171352_0001
skip existing: 20240423-171352_0002
skip existing: 20240423-171352_0003
skip existing: 20240423-171352_0004
skip existing: 20240423-171352_0005
skip existing: 20240423-171352_0006
skip existing: 20240423-171352_0007
skip existing: 20240423-171352_0008
skip existing: 20240423-171352_0009
skip existing: 20240423-171352_0010
skip existing: 20240423-171352_0011
skip existing: 20240423-171352_0012
skip existing: 20240423-171352_0013
skip existing: 20240423-1713

In [32]:
import subprocess
from pathlib import Path

audio_chan = 2
manifest_path = patient_dir / f"{patient}_vad_merged_intervals.csv"

logs_dir = patient_dir / "transcription_slurm_logs"
batch_dir = patient_dir / "transcription_slurm_scripts"
logs_dir.mkdir(parents=True, exist_ok=True)
batch_dir.mkdir(parents=True, exist_ok=True)

extract_worker = "/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/audio_worker.py"
python_bin = "/scratch/tahaismail424/miniforge3/envs/whisper_env/bin/python3"
whisperx_bin = "/scratch/tahaismail424/miniforge3/envs/whisper_env/bin/whisperx"
ffmpeg_lib_dir = "/scratch/tahaismail424/miniforge3/envs/whisper_env/lib"

whisper_model = "large-v2"
whisper_chunk_size = 20

def pick_whisper_resources(duration_s: float):
    if duration_s <= 60:
        return "mps:l40:16", 8
    if duration_s <= 300:
        return "mps:l40:20", 6
    if duration_s <= 900:
        return "mps:l40:24", 4
    return "mps:l40:24", 2

partition = "guppy"
conda_activate = "source /scratch/tahaismail424/.bash_profile && conda activate whisper_env"

submitted = []
failed = []

for row_idx, row in merged_df.iterrows():
    interval_id = row["interval_id"]
    duration_s = float(row.get("duration_s", 0.0))
    whisper_gres, whisper_batch_size = pick_whisper_resources(duration_s)
    transcription_dir = Path(write_dir) / patient / "vad_data" / interval_id / "transcription"
    success_path = transcription_dir / "_SUCCESS"

    if success_path.exists():
        print(f"skip existing: {interval_id}")
        continue

    sbatch_text = f"""#!/bin/bash
#SBATCH --job-name=whx_{interval_id}
#SBATCH --partition={partition}
#SBATCH --gres={whisper_gres}
#SBATCH --cpus-per-task=4
#SBATCH --mem=32G
#SBATCH --qos=default_tier
#SBATCH --time=24:00:00
#SBATCH --output={logs_dir}/{interval_id}_%j.out
#SBATCH --error={logs_dir}/{interval_id}_%j.err

set -eo pipefail

{conda_activate}

export HF_HOME='/scratch/tahaismail424/hf'
export TRANSFORMERS_OFFLINE=1
export HF_HUB_OFFLINE=1
export HF_DATASETS_OFFLINE=1
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

INTERVAL_DIR="{write_dir}/{patient}/vad_data/{interval_id}"
TRANSCRIPTION_DIR="$INTERVAL_DIR/transcription"
WHISPERX_DIR="$TRANSCRIPTION_DIR/whisperx"
AUDIO_PATH="$TRANSCRIPTION_DIR/audio.wav"
SUCCESS_PATH="$TRANSCRIPTION_DIR/_SUCCESS"
ERROR_PATH="$TRANSCRIPTION_DIR/transcription_error.txt"

mkdir -p "$TRANSCRIPTION_DIR"
mkdir -p "$WHISPERX_DIR"

echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "INTERVAL_ID: {interval_id}"
echo "ROW_IDX: {row_idx}"
echo "DURATION_S: {duration_s}"
echo "GRES: {whisper_gres}"

# clear stale markers
rm -f "$SUCCESS_PATH"
rm -f "$ERROR_PATH"

# ---------------------------
# 1) Extract audio
# ---------------------------
{python_bin} "{extract_worker}" "{manifest_path}" "{row_idx}" "{audio_chan}"

if [[ ! -f "$AUDIO_PATH" ]]; then
    echo "ERROR: audio extraction did not produce $AUDIO_PATH" | tee "$ERROR_PATH"
    exit 1
fi

if [[ ! -s "$AUDIO_PATH" ]]; then
    echo "ERROR: extracted audio file is empty: $AUDIO_PATH" | tee "$ERROR_PATH"
    exit 1
fi

# ---------------------------
# 2) Run WhisperX
# ---------------------------
echo "CONDA_PREFIX: $CONDA_PREFIX" > "$WHISPERX_DIR/log.txt"
echo "which python: $(which python)" >> "$WHISPERX_DIR/log.txt"
echo "which whisperx: $(which whisperx)" >> "$WHISPERX_DIR/log.txt"
echo "whisperx_bin: {whisperx_bin}" >> "$WHISPERX_DIR/log.txt"
echo "duration_s: {duration_s}" >> "$WHISPERX_DIR/log.txt"
echo "gres: {whisper_gres}" >> "$WHISPERX_DIR/log.txt"
echo "LD_LIBRARY_PATH: $LD_LIBRARY_PATH" >> "$WHISPERX_DIR/log.txt"

"{whisperx_bin}" "$AUDIO_PATH" \\
    --model {whisper_model} \\
    --device cuda \\
    --align_model WAV2VEC2_ASR_LARGE_LV60K_960H \\
    --diarize \\
    --highlight_words True \\
    --output_dir "$WHISPERX_DIR" \\
    --language English \\
    --model_cache_only True \\
    >> "$WHISPERX_DIR/log.txt" 2>&1

WHISPERX_STATUS=$?

if [[ $WHISPERX_STATUS -ne 0 ]]; then
    echo "ERROR: whisperx failed with exit code $WHISPERX_STATUS" | tee "$ERROR_PATH"
    exit $WHISPERX_STATUS
fi

# ---------------------------
# 3) Validate WhisperX outputs
# ---------------------------
JSON_COUNT=$(find "$WHISPERX_DIR" -maxdepth 1 -type f \\( -name "*.json" -o -name "*.srt" -o -name "*.vtt" -o -name "*.tsv" -o -name "*.txt" \\) | wc -l)

if [[ "$JSON_COUNT" -eq 0 ]]; then
    echo "ERROR: whisperx finished but no output files were found in $WHISPERX_DIR" | tee "$ERROR_PATH"
    exit 1
fi

# ---------------------------
# 4) Mark success only now
# ---------------------------
echo "ok" > "$SUCCESS_PATH"

echo "END: $(date)"
"""

    sbatch_path = batch_dir / f"{interval_id}.sbatch"
    sbatch_path.write_text(sbatch_text)

    try:
        res = subprocess.run(
            ["sbatch", str(sbatch_path)],
            capture_output=True,
            text=True,
            check=True
        )
        submitted.append((interval_id, res.stdout.strip()))
        print(f"submitted: {interval_id} -> {res.stdout.strip()}")
    except subprocess.CalledProcessError as e:
        failed.append((interval_id, e.stderr.strip()))
        print(f"FAILED SUBMISSION: {interval_id}")
        print(e.stderr)

print(f"submitted={len(submitted)}, failed={len(failed)}")

submitted: 20240423-124841_0000 -> Submitted batch job 206759
submitted: 20240423-124841_0001 -> Submitted batch job 206760
submitted: 20240423-124841_0002 -> Submitted batch job 206761
submitted: 20240423-124841_0003 -> Submitted batch job 206762
submitted: 20240423-124841_0004 -> Submitted batch job 206763
submitted: 20240423-124841_0005 -> Submitted batch job 206764
submitted: 20240423-124841_0006 -> Submitted batch job 206765
submitted: 20240423-124841_0007 -> Submitted batch job 206766
submitted: 20240423-124841_0008 -> Submitted batch job 206767
submitted: 20240423-124841_0009 -> Submitted batch job 206768
submitted: 20240423-124841_0010 -> Submitted batch job 206769
submitted: 20240423-124841_0011 -> Submitted batch job 206770
submitted: 20240423-124841_0012 -> Submitted batch job 206771
submitted: 20240423-171352_0000 -> Submitted batch job 206772
submitted: 20240423-171352_0001 -> Submitted batch job 206773
submitted: 20240423-171352_0002 -> Submitted batch job 206774
submitte